# Fine-tuning do BERTimbau — avaliações de delivery

**Antes de rodar:** Runtime → Alterar tipo de ambiente de execução → **GPU (T4)**.
Requer o dataset publicado no Hub (`nimrqz/delivery-reviews`) e um token do Hugging Face em *Secrets* (🔑) com o nome `HF_TOKEN`.

In [ ]:
!git clone https://github.com/nimrqz/delivery-reviews.git
%cd delivery-reviews
!pip install -q -e ".[train]" 

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_USERNAME"] = "nimrqz"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (vai ficar lento!)")

In [ ]:
!python scripts/05_train.py --hf-dataset nimrqz/delivery-reviews --epochs 5

## Resultados

In [ ]:
import json
m = json.load(open("outputs/metrics.json"))
for t in ("sentimento", "urgencia", "problemas"):
    print(f"{t:11s} F1 micro={m[t]['f1_micro']:.3f}  macro={m[t]['f1_macro']:.3f}")
print(json.dumps(m["problemas"]["f1_per_label"], indent=2))

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from delivery_reviews.labels import SENTIMENTS, URGENCIES
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (task, names) in zip(axes, [("sentimento", SENTIMENTS), ("urgencia", URGENCIES)]):
    cm = np.array(m[task]["confusion_matrix"]); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(names)), names); ax.set_yticks(range(len(names)), names)
    ax.set_title(task); ax.set_xlabel("previsto"); ax.set_ylabel("real")
    for i in range(len(names)):
        for j in range(len(names)): ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout(); plt.savefig("outputs/confusion_matrices.png", dpi=150); plt.show()

## Publicar no Hub (opcional)

In [ ]:
!python scripts/06_push_model.py